# Op³ quickstart -- 5 minutes

This notebook walks through the canonical Op³ workflow on the NREL 5 MW OC3 monopile, exercising every major subsystem in under 100 lines of code.

1. Build a fixed-base TowerModel and run eigenvalue analysis
2. Replace the foundation with a PISA-derived 6x6 stiffness
3. Propagate soil parameter uncertainty through Monte Carlo
4. Calibrate the tower EI scale factor against the published reference via grid Bayesian inversion
5. Export an OpenFAST SoilDyn input file from the same matrix

## 1. Fixed-base eigenvalue

In [ ]:
from op3 import build_foundation, compose_tower_model

model = compose_tower_model(
    rotor='nrel_5mw_baseline',
    tower='nrel_5mw_oc3_tower',
    foundation=build_foundation(mode='fixed'),
)
freqs = model.eigen(n_modes=3)
print(f'Op3 fixed-base f1 = {freqs[0]:.4f} Hz   (Jonkman & Musial 2010 OC3 reference: 0.275 Hz)')

## 2. PISA-derived 6x6 foundation

In [ ]:
from op3.foundations import foundation_from_pisa
from op3.standards.pisa import SoilState

profile = [
    SoilState(0.0,  5.0e7, 35, 'sand'),
    SoilState(15.0, 1.0e8, 35, 'sand'),
    SoilState(36.0, 1.5e8, 36, 'sand'),
]
fnd = foundation_from_pisa(diameter_m=6.0, embed_length_m=36.0, soil_profile=profile)
K = fnd.stiffness_matrix
print(f'PISA Kxx   = {K[0,0]:.3e} N/m')
print(f'PISA Krxrx = {K[3,3]:.3e} Nm/rad')

model_pisa = compose_tower_model(
    rotor='nrel_5mw_baseline',
    tower='nrel_5mw_oc3_tower',
    foundation=fnd,
)
f1_pisa = model_pisa.eigen(n_modes=3)[0]
print(f'f1 with PISA flex = {f1_pisa:.4f} Hz')

## 3. Soil uncertainty propagation

In [ ]:
from op3.uq import SoilPrior, propagate_pisa_mc, summarise_samples

priors = [
    SoilPrior(0.0,  5.0e7, G_cov=0.30, soil_type='sand'),
    SoilPrior(15.0, 1.0e8, G_cov=0.30, soil_type='sand'),
    SoilPrior(36.0, 1.5e8, G_cov=0.30, soil_type='sand'),
]
samples = propagate_pisa_mc(diameter_m=6.0, embed_length_m=36.0,
                            soil_priors=priors, n_samples=300)
summary = summarise_samples(samples)
for k in ['Kxx', 'Krxrx']:
    s = summary[k]
    print(f"{k:<7} mean={s['mean']:.3e}   COV={s['cov']:.2f}   p05/p95=[{s['p05']:.3e}, {s['p95']:.3e}]")

## 4. Bayesian EI calibration vs the published reference

In [ ]:
import numpy as np
from op3.uq import grid_bayesian_calibration, normal_likelihood
from op3.opensees_foundations import tower_loader as TL
from dataclasses import replace

def forward(scale):
    orig = TL.load_elastodyn_tower
    def patched(*a, **kw):
        t = orig(*a, **kw)
        return replace(t, ei_fa_Nm2=t.ei_fa_Nm2*scale, ei_ss_Nm2=t.ei_ss_Nm2*scale)
    TL.load_elastodyn_tower = patched
    try:
        m = compose_tower_model(
            rotor='nrel_5mw_baseline',
            tower='nrel_5mw_oc3_tower',
            foundation=build_foundation(mode='fixed'),
        )
        return float(m.eigen(n_modes=3)[0])
    finally:
        TL.load_elastodyn_tower = orig

post = grid_bayesian_calibration(
    forward_model=forward,
    likelihood_fn=normal_likelihood(measured=0.2766, sigma=0.01),
    grid=np.linspace(0.7, 1.3, 21),
)
print(f'EI scale posterior:  mean = {post.mean:.3f} +/- {post.std:.3f}')
print(f'                     5%-95% = [{post.p05:.3f}, {post.p95:.3f}]')

## 5. Export to OpenFAST SoilDyn

In [ ]:
from op3.openfast_coupling.soildyn_export import write_soildyn_from_pisa

out = write_soildyn_from_pisa(
    'demo_SoilDyn.dat',
    diameter_m=6.0, embed_length_m=36.0,
    soil_profile=profile,
    location_xyz=(0.0, 0.0, -45.0),
)
print(f'wrote {out}')
print('Plug into your .fst as:  CompSoil = 1   SoilFile = "demo_SoilDyn.dat"')

## What you just did

You exercised every Op³ subsystem in under 100 lines:

| Step | Module |
|---|---|
| 1 | `op3.composer`, `op3.opensees_foundations.builder` |
| 2 | `op3.standards.pisa`, `op3.foundations.foundation_from_pisa` |
| 3 | `op3.uq.propagation` |
| 4 | `op3.uq.bayesian` |
| 5 | `op3.openfast_coupling.soildyn_export` |

Next: see the other tutorials for end-to-end DLC simulation, Mode D dissipation weighting, and the full DNV / IEC conformance audit.